<a href="https://colab.research.google.com/github/ceuratfmg2mai/fakereviews/blob/main/05.%20evaluacion_resultados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Categorización de reseñas con el uso de una base de datos vectorial aplicando la técnica de similitud

En esta etapa del TFM, emplearemos una base de datos vectorial con el objetivo de aplicar técnicas de búsqueda por similitud. Esto nos permitirá analizar la relevancia de las reseñas recuperadas y examinar la coherencia entre su similitud textual y su clasificación preexistente ('Fake' o 'Genuine').

Para ello, utilizaremos la base de datos vectorial Qdrant con el fin de encontrar reseñas de Yelp que sean textualmente parecidas entre sí, según la interpretación de su significado realizada por un modelo de embeddings. Cabe destacar que estas reseñas ya fueron clasificadas previamente por humanos como 'Fake' o 'Genuine' y esta clasificación será almacenada en Qdrant.

Al introducir nuevas reseñas como consulta en Qdrant, compararemos los hallazgos de la plataforma (*es decir, las reseñas que Qdrant identifica como semánticamente similares, mostrando la clasificación ('Fake' o 'Genuine')*) con las evaluaciones realizadas por humanos. Estas evaluaciones humanas definirán tanto la relevancia de una reseña para la consulta específica como su verdadera clasificación ('Fake' o 'Genuine').

Este cruce permitirá no solo medir la efectividad de Qdrant para identificar contenido textualmente relevante, sino también evaluar la precisión de las clasificaciones originales de DeepSeek sobre dicho contenido semánticamente similar. Así, se podrán revelar patrones y posibles áreas de mejora tanto en el sistema de búsqueda por similitud como en el proceso de clasificación de reseñas.

# Desarrollo
Este notebook estará compuesto de los siguientes principales pasos:
1. Inicialización de variables.
2. Envío de reseñas categorizadas por humanos a qdrant.

# Configuración

## Instalación

In [28]:
!pip install -U langchain-community  > /dev/log 2>&1
!pip install -U qdrant-client > /dev/log 2>&1

## Librerías

In [56]:
import polars as pl
import pandas as pd
import pyarrow
import matplotlib.pyplot as plt
plt.style.use('Solarize_Light2')
import seaborn as sns
import time
start_time_global = time.time()
import os
import sys
from google.colab import userdata
from google.colab import drive
from importlib import metadata
import uuid

from sklearn.metrics import cohen_kappa_score
color = '\033[1m\033[38;5;208m'
print(f"{color}Versión pandas: {pd.__version__}")
print(f"{color}Versión polars: {pl.__version__}")
print(f"{color}Versión pyarrow: {pyarrow.__version__}")


Versión pandas: 2.2.2
Versión polars: 1.21.0
Versión pyarrow: 18.1.0


## Inicialización de las variables de entorno

In [30]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from qdrant_client import QdrantClient, models
print(f"{color}Versión langchain: {metadata.version('langchain')}")
print(f"{color}Versión qdrant_client: {metadata.version('qdrant_client')}")
# 1. Configurar Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2") # Generates vector embeddings for each chunk
# Proba el modelo y pedir que genere el vector para la palabra test
# Número de elementos que contiene esa lista.
vector_size_test = len(embedding_model.embed_query("test"))
print(vector_size_test)

Versión langchain: 0.3.25
Versión qdrant_client: 1.14.2


<ipython-input-30-c9351a180a46>:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2") # Generates vector embeddings for each chunk
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommend

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384


In [31]:
drive.mount('/content/drive', force_remount=True)
explicit_work_path = '/content/drive/MyDrive/Colab Notebooks/tfm_grupo_2/'
print(os.listdir(explicit_work_path))

Mounted at /content/drive
['base_reviews.json', 'yelp_academic_dataset_review_selected_0_255.jsonl', 'final_reviews_categorize_0_255.jsonl', 'data_yelp', 'yelp_academic_dataset_review_selected.jsonl', 'yelp_academic_dataset_review_selected.arrow', 'yelp_academic_dataset_review_selected.csv', 'yelp_academic_dataset_review_selected_prompt.arrow', 'yelp_academic_dataset_review_selected_prompt.jsonl', 'final_reviews_categorize.jsonl', 'yelp_academic_dataset_review_final_selected_1900.jsonl', 'yelp_academic_dataset_review_golden_evaluador_1.jsonl', 'ground_truth.jsonl', 'yelp_academic_dataset_review_golden_evaluador_2.json']


In [32]:
try:
    qdrant_key = userdata.get("QDRANT_KEY")
    qdrant_url = 'https://73e55abe-f0f4-4f08-b30d-c1e2187ebc45.europe-west3-0.gcp.cloud.qdrant.io:6333'
    print("Datos de Qdrant cargados.")
except KeyError:
    print("Error: Las variables de entorno.")

Datos de Qdrant cargados.


In [33]:
review_categorice_file_path = f'{explicit_work_path}ground_truth.jsonl'
df_data_ground_truth = pl.read_json(review_categorice_file_path)
print(f"\nTotal de reseñas categorizadas: {df_data_ground_truth.shape[0]}")


Total de reseñas categorizadas: 100


In [34]:
start_time=time.time()
# Lectura del fichero de las reseñas seleccionadas de yelp de 256 a 512
review_selected_file_path = f'{explicit_work_path}yelp_academic_dataset_review_selected.jsonl'
df_data_reviews_selected_pl = pl.read_json(review_selected_file_path)
print(f"\nTotal de reseñas seleccionadas con longitud de 256 a 512: {df_data_reviews_selected_pl.shape[0]}")
# Lectura del fichero de las reseñas seleccionadas de yelp de 0 a 255
review_selected_file_path = f'{explicit_work_path}yelp_academic_dataset_review_selected_0_255.jsonl'
df_data_reviews_selected_0_255_pl = pl.read_json(review_selected_file_path)
print(f"\nTotal de reseñas seleccionadas con longitud de 0 a 255: {df_data_reviews_selected_0_255_pl.shape[0]}")
# Lectura del fichero de las reseñas categorizadas de yelp de 255 a 512
review_categorice_file_path = f'{explicit_work_path}final_reviews_categorize.jsonl'
df_data_reviews_categorice_pl = pl.read_ndjson(review_categorice_file_path)
print(f"\nTotal de reseñas categorizadas con longitud de 256 a 512: {df_data_reviews_categorice_pl.shape[0]}")
# Lectura del fichero de las reseñas categorizadas de yelp 0 a 255
review_categorice_0_255_file_path = f'{explicit_work_path}final_reviews_categorize_0_255.jsonl'
df_data_reviews_categorice_0_255_pl = pl.read_ndjson(review_categorice_0_255_file_path)
print(f"\nTotal de reseñas categorizadas con longitud de 0 a 255: {df_data_reviews_categorice_0_255_pl.shape[0]}")

try:
    columns_target_classification = ['review_id','classification']
    columns_target_selected = ['review_id','user_id','business_id','text','stars','categories','review_count_business','stars_business',
        'review_count_user','average_stars_user']
    df_final_fake_genuine_pl = pl.concat([
        df_data_reviews_categorice_pl.select(columns_target_classification),
        df_data_reviews_categorice_0_255_pl.select(columns_target_classification)
    ])
    df_final_selected_pl = pl.concat([
        df_data_reviews_selected_0_255_pl.select(columns_target_selected),
        df_data_reviews_selected_pl.select(columns_target_selected)
    ])
    df_final_pl = df_final_selected_pl.join(
        df_final_fake_genuine_pl,
        on='review_id', # La columna común para la unión
        how='left'
    ).select(
            [
                pl.col('review_id'),
                pl.col('user_id'),
                pl.col('business_id'),
                pl.col('text'),
                pl.col('stars'),
                pl.col('categories'),
                pl.col('review_count_business'),
                pl.col('stars_business'),
                pl.col('review_count_user'),
                pl.col('average_stars_user'),
                pl.col('classification')
            ]
        ).filter(pl.col('classification') != 'null')
except Exception as e:
    print(f"An error occurred: {e}")
else:
    print(f"La operación fue exitosa en {time.time()-start_time:.2f}seg")
    print("-" * 50 + "\n")
    print(f'Total reseñas: {len(df_final_pl)}')



Total de reseñas seleccionadas con longitud de 256 a 512: 17000

Total de reseñas seleccionadas con longitud de 0 a 255: 16999

Total de reseñas categorizadas con longitud de 256 a 512: 24259

Total de reseñas categorizadas con longitud de 0 a 255: 16999
La operación fue exitosa en 0.31seg
--------------------------------------------------

Total reseñas: 29341


In [35]:
df_human = df_data_ground_truth.join(
        df_final_pl.drop("classification"),
        on='review_id', # La columna común para la unión
        how='inner'
    ).select(
            [
                pl.col('review_id'),
                pl.col('user_id'),
                pl.col('business_id'),
                pl.col('text'),
                pl.col('stars'),
                pl.col('categories'),
                pl.col('review_count_business'),
                pl.col('stars_business'),
                pl.col('review_count_user'),
                pl.col('average_stars_user'),
                pl.col('classification')
            ]
        ).filter(pl.col('classification') != 'null')

Variables qdrant

In [36]:
# Nombre de la conlección
COLLECTION_NAME = "tfm2g"

In [37]:
# --- Inicialización del cliente de Qdrant ---
qdrant_cloud_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_key,
    timeout=30
)

In [38]:
# Agrega la carpeta fakereviews al syspath del proyecto
if explicit_work_path not in sys.path:
    sys.path.append(explicit_work_path)
!wget https://raw.githubusercontent.com/ceuratfmg2mai/fakereviews/refs/heads/main/qdrant_connection.py
from qdrant_connection import QdrantDataIngestor

--2025-05-19 12:12:46--  https://raw.githubusercontent.com/ceuratfmg2mai/fakereviews/refs/heads/main/qdrant_connection.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14645 (14K) [text/plain]
Saving to: ‘qdrant_connection.py’

qdrant_connection.p 100%[===================>]  14.30K  --.-KB/s    in 0.001s  

2025-05-19 12:12:46 (10.0 MB/s) - ‘qdrant_connection.py’ saved [14645/14645]



In [39]:
ingestor = QdrantDataIngestor(
        qdrant_url=qdrant_url,
        qdrant_key=qdrant_key,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        vector_size=vector_size_test
    )

In [40]:
payload_fields_to_index = {
        "review_id": models.PayloadSchemaType.KEYWORD,
        "user_id": models.PayloadSchemaType.KEYWORD,
        "business_id": models.PayloadSchemaType.KEYWORD,
        "stars": models.PayloadSchemaType.INTEGER,
        "categories": models.PayloadSchemaType.KEYWORD,
        "stars_business": models.PayloadSchemaType.FLOAT,
        "review_count_business": models.PayloadSchemaType.INTEGER,
        "review_count_user": models.PayloadSchemaType.INTEGER,
        "average_stars_user": models.PayloadSchemaType.FLOAT,
        "classification": models.PayloadSchemaType.KEYWORD
    }

In [41]:
ingestor.setup_collection(payload_fields_to_index=payload_fields_to_index)

La colección 'tfm2g' ya existe.
Configurando índices de payload...
Índice de payload creado/verificado para el campo: review_id
Índice de payload creado/verificado para el campo: user_id
Índice de payload creado/verificado para el campo: business_id
Índice de payload creado/verificado para el campo: stars
Índice de payload creado/verificado para el campo: categories
Índice de payload creado/verificado para el campo: stars_business
Índice de payload creado/verificado para el campo: review_count_business
Índice de payload creado/verificado para el campo: review_count_user
Índice de payload creado/verificado para el campo: average_stars_user
Índice de payload creado/verificado para el campo: classification
Configuración de la colección completada.


# Envío de datos categorizados por Humanos a Qdrant

In [42]:
ingestor.ingest_dataframe(df_human)


Iniciando ingestión de 89 filas en la colección 'tfm2g'...
Lote 1/1 (filas aprox. 1-89) enviado. 89 puntos.
Progreso: 89/89 filas procesadas. Tiempo total: 4.45 seg.

Ingestión completada para 89 filas.
Tiempo total de ingestión: 4.45 segundos.
Todos los lotes fueron procesados sin errores reportados por el cliente.


Realizamos la verficación de la inserción

In [43]:
ingestor.verify_ingestion(original_id_to_check = '78CkRZ7RTAzHSWj8T6Cwfg')


--- Verificación de Datos Insertados ---
Número total de puntos en la colección 'tfm2g': 178
Intentando recuperar la reseña con 'review_id' (en payload) igual a: '78CkRZ7RTAzHSWj8T6Cwfg'
Punto encontrado por 'review_id' en payload.

--- Detalles del Punto de Muestra ---
  ID de Qdrant (UUID generado): 8dc30edd-a282-439e-a354-0a61004e3d5e
  Payload:
    review_id: 78CkRZ7RTAzHSWj8T6Cwfg
    review: I'd love to give a 3 1/2 Star. Really pleasant staff, very accommodating and friendly.  Food was good.
    user_id: PzKx-e5Prx-OiSOImGhnVg
    business_id: _FQ-nrR--8Q1O1sYxefNug
    stars: 3
    categories: ['Italian', 'Pizza', 'Restaurants']
    review_count_business: 19
    stars_business: 3.5
    review_count_user: 6
    average_stars_user: 4.14
    classification: Genuine


In [46]:
ingestor.get_classification_by_review_id(review_id = '78CkRZ7RTAzHSWj8T6Cwfg')

'Genuine'

In [61]:
df_human_deepseek_pl = df_data_ground_truth.join(df_final_pl, on="review_id", how="inner")
list_classification_human_actual_predicted = df_human_deepseek_pl[f"classification"].to_list()
list_classification_deepseek_predicted = df_human_deepseek_pl[f"classification_right"].to_list()


In [60]:
kappa = cohen_kappa_score(list_classification_human_actual_predicted, list_classification_deepseek_predicted)
print(f"\n--- Fiabilidad Inter-evaluador ---")
print(f"Kappa de Cohen: {kappa:.4f}")

# Interpretación (igual que en el ejemplo anterior)
if kappa < 0: interpretacion = "Pobre acuerdo."
elif kappa == 0: interpretacion = "Acuerdo equivalente al azar."
elif kappa < 0.21: interpretacion = "Ligero acuerdo."
elif kappa < 0.41: interpretacion = "Aceptable acuerdo."
elif kappa < 0.61: interpretacion = "Moderado acuerdo."
elif kappa < 0.81: interpretacion = "Sustancial acuerdo."
else: interpretacion = "Casi perfecto o perfecto acuerdo."
print(f"Interpretación: {interpretacion}")

acuerdo_simple_porcentaje = sum(1 for a, b in zip(list_classification_human_actual_predicted, list_classification_deepseek_predicted) if a == b) / len(list_classification_human_actual_predicted) * 100
print(f"Porcentaje de acuerdo simple (observado): {acuerdo_simple_porcentaje:.2f}%")



--- Fiabilidad Inter-evaluador ---
Kappa de Cohen: 0.2645
Interpretación: Aceptable acuerdo.
Porcentaje de acuerdo simple (observado): 60.67%
